# Projeto PCD - Treinando pygame com jogo de plataforma

## Importando o que for necessário

In [184]:
import pygame as pg
import sys
import random

## Cores

In [185]:
white = (255, 255, 255)
red = (255, 0, 0)
green = (0, 255, 0)
blue = (0, 0, 255)
black = (0, 0, 0)

## Código

### Definindo coisas

Nesta parte, são definidas algumas coisas importantes para a funcionalidade do código:

Constantes:
> * `WIDGHT`: Define a largura da tela;<br>
> * `HEIGHT`: Define a altura da tela;<br>
> * `ACC`: Define a aceleração vertical do player para movimentação mais realista;<br>
> * `FRIC`: Define um atrito simples para o movimento do player;<br>
> * `FPS`: Define a taxa de frames por segundo do jogo;<br>

O que está abaixo de "Criando fontes" serve para definir quais serão as fontes e tamanhos usadas:

> * `font`: Define uma fonte padrão do tipo "Verdana" de tamanho 60;<br>
> * `small_font`: Define uma fonte do mesmo tipo de `font`, mas com tamanho 20;<br>

A linha 32 define o nome do jogo, enquanto a linha 20 define o objeto vec, que é um vetor 2d, que será usado em diversas outras ocasiões, como para definir aceleração, veocidade, etc.

In [186]:
####################################################
#----------------------Setup-----------------------#
####################################################

pg.init()

####################################################

##############
# Constantes #
##############
WIDTH = 400
HEIGHT = 450
ACC = 0.5
FRIC = -0.12
FPS = 60

####################################################

vec = pg.math.Vector2

####################################################

##################
# Criando fontes #
##################
font = pg.font.SysFont("Verdana", 60)
small_font = pg.font.SysFont("Verdana", 20)

#########################################################

pg.display.set_caption('The Platform Game')

### Sprites

Esta parte defini tudo sobre as sprites usadas no jogo. Todas elas foram feitas usando classes, uma maneira mais simples de definir um certo objeto. Para tal, todas as classes são "filhas" da classe Sprite do pygame, que tem alguma das funções aqui usadas.

As sprites usadas e suas respecticas funções são:

#### `Player`
Essa classe define o jogador como um todo, dizendo como ele deve se mover, como deve pular, quando não deve, etc. Suas funções são:<br>

##### `__init(self, game)__`

Essa função caracteriza o jogador, definindo sua imagem, sua delimitação, posição - aqui foi usado, inclusive, o objeto vec, para poder facilitar a movimentação e deixar o jogo melhor, definindo sua posição como um vetor no plano da tela - que pode ser alterada facilmente, velocidade, aceleração e vida.

Tudo que é definido nessa função, não necessáriamente será modificado ou usado diretamente nela, pois ela só define suas variáveis principais. Para tal, foram utilizados dois argumentos, `self` e `game`. O argumento `self` é usado para definir variáveis que podem ser usadas em toda a classe, fazendo com que tudo que for criado em tal função, possa ser usado em outras funções - e até mesmo em outras classes. Já o argumento `game` é usado para definir `self.game = game`, que chama a classe `Game`, para que tal classe possa acessar variáveis contidas em `Game`, como, por exemplo, `self.game.platforms`, que chama o conjunto de todas as sprites do tipo plataforma contidas no jogo, enquanto ele roda.

Vars:
> * `self.surf`: Irá criar a superfície do jogador, que terá dimensões de 30x30 px. `self.surf.fill()` irá dar cor a essa superfície através de coordenadas RGB. A cor `(0, 255, 255)` dará aspecto ciano ao personagem, que será um quadrado simples.<br>
> * `self.rect`: Define a caixa delimitadora do player, usada para verificar colisões, principalmente. Para tal, foi usado `self.surf.get_rect()`, que encontra a caixa mais "justa" para o objeto e o posiciona na tela.<br>
> * `self.pos`, um vetor `(x, y)`: Tal variável é um vetor, que define onde o jogador está na tela. Para deixar o jogo mais simples e intuitivo, a superfície - ou a imagem - do jogador, assim como sua caixa delimitadora, serão movidos a partir desse vetor, permitindo que, ao invés de modificar diretamente seu `self.rect`, modifiquemos seu vetor posição, que é mais simples e versátil de ser usado.<br>
> * `self.vel`, um vetor `(x, y)`: Este define o vetor de velocidade do player. Geralmente será a partir dele que sua posição será modificada.<br>
> * `self.acc`, um vetor `(x, y)`: Esta variável irá definir a aceleração do jogoador.<br>
> * `self.jumping`: Isso irá dizer se o jogador está pulando ou não.<br>
> * `self.life`: A vida do jogador.

##### `move(self)`

Como o próprio nome diz, tal função define toda a movimentação do jogador. Essa função também usa o argumento self, permitindo que ela acesse todas as variáveis com esse atributo na classe e permitindo que qualquer outra função acesse suas variáveis.

Para tal, primeiramente é difinida uma aceleração inicial de `(0, 0.5)`, ou seja, uma aceleração positiva na direção `y`. Isso é feito para poder simular uma gravidade. Depois, é usado um algorítmo que reconhece uma interação do usuário com o computador, o pressionamento de uma tecla. Com isso, o código define que, se o usuário está apertando a tecla "left" ou "a", a aceleração na direção `x` é `-ACC`, ou seja, para a esquerda. Porém, se o usuário estiver apertando a tecla "right" ou "d", o código define a aceleração na direção `x` como sendo `ACC`, ou seja, para a direita.

Em

> ```
> 31  self.acc.x += self.vel.x * FRIC
> 32  self.vel += self.acc
> 33  self.pos += self.vel + 0.5 * self.acc
> ```

é definida a movimentação real do jogador, onde, na linha 31, é adicionado o atrito dinâmico do ambiente, na linha 32 a velocidade é atualizada de acordo com a aceleração - tanto em `x`, como em `y` - e a linha 33 define a posição a partir de uma equação simples.

Já o bloco

> ```
> 42  if self.pos.x > WIDTH:
> 43      self.pos.x = 0
> 44  if self.pos.x < 0:
> 45      self.pos.x = WIDTH
> ```

Cria um efeito de "teletransporte do jogoador caso ele ultrapasse o limite da tela, fazendo com que ele se "teletransporte" para o outro lado, como no jogo Pac-Man.

A linha 47 define o `self.rect` do jogador como a posição `self.pos` dele.|

##### `jump(self)`

Define o pulo do jogador.

Para tal, a função usa de `pg.sprite.spritecollide()` para verificar se o player está encima de uma plataforma e a variável booleana `self.jumping`:

> ```
> 50  hits = pg.sprite.spritecollide(
> 51      self,
> 52      self.game.platforms,
> 53      False
> 54  )
> ```

`hits` verifica se o player está encima de uma plataforma ou não e em qual está.

> ```
> 55  if hits and not self.jumping:
> 56      self.jumping = True
> 57
> 58      self.vel.y = -15
> ```

Este bloco verifica se o player já está pulando e se ele está encima de algo - porque não faz sentido o player pular enquanto cai e, nesse caso, nem dar um pulo duplo. Caso as o player esteja encima de uma plataforma e não esteja pulando, ele define `self.jumping = True`, para dizer ao código que ele está no meio de um pulo, e define `self.vel.y = -15` fazendo com que o player suba. Note que, por conta do efeito de gravidade que foi colocado, mesmo definindo um valor para a velocidade, o player não sobe a velocidade constante e, após parar de pular, ele cai como que em queda livre.

##### `cancel_jump(self)`

Cancela o pulo - acho que estava meio óbvio... mas é sempre bom dizer.

Essa função define como o pulo irá ser cancelado de maneira muito simples:

> ```
> 61  if self.jumping:
> 62      if self.vel.y < -3:
> 63          self.vel.y = -3
> ```

Com isso, quando a função é chamada, caso o jogador esteja em pulo - porque, de novo, não faz sentido cancelar um pulo que não existe - a função define `self.vel.y = -3`, fazendo com que, caso o player ainda esteja subindo, ele tenha uma velocidade de pulo muito baixa, o que faz com que a gravidade faça-o descer.

##### `update`

Essa função básicamente faz com que o player não atravesse as plataformas.

Para funcionar, essa função usa um conceito parecido com a função `jump()`, usando de colisão:

> ```
> 67  hits = pg.sprite.spritecollide(
> 68      self,
> 69      self.game.platforms,
> 70      False
> 71  )
> ```

Ou seja, aqui ele, de novo, verifica se o jogador está ou não colidindo com uma plataforma, porém essa função também usa as informações da plataforma que está colidindo com o player.

> ```
> 73  if self.vel.y > 0 and hits:
> 74      platform = hits[0]
> 75
> 76      if self.pos.y < platform.rect.bottom:
> 77          self.vel.y = 0
> 78          self.pos.y = platform.rect.top + 1
> 79          self.jumping = False
> 80          self.pos.x += platform.vel.x*platform.direction
> ```

Primeiro esse bloco verifica se o player está caindo E está colidindo com uma plataforma. Caso isso seja verdadeiro, ele defini a plataforma que está colidindo com `platform = hits[0]`. Então, caso `self.pos.y < platform.rect.bottom`, o código define sua velocidade em `y` como nula, sua posição em y como o topo da plataforma +1 - o +1 serve para que o player não fique colidindo eternamente com a plataforma -, diz para o código que ele parou de pular e, por fim, faz com que o player se mova junto da plataforma, para que ele não caia quando ela se movimentar.

##### `draw(self)`

Essa função só desenha o player na tela.

##### Código

In [187]:
##########
# Player #
##########
class Player(pg.sprite.Sprite):
    def __init__(self, game):
        super().__init__()
        self.game = game

        self.surf = pg.Surface((30, 30))
        self.surf.fill((0, 255, 255))
        
        self.rect = self.surf.get_rect()

        self.pos = vec((10, 360))
        self.vel = vec(0,0)
        self.acc = vec(0,0)

        self.jumping = False

        self.life = 100
    
    def move(self):
        self.acc = vec(0, 0.5)
 
        pressed_keys = pg.key.get_pressed()
            
        if pressed_keys[pg.K_LEFT] or pressed_keys[pg.K_a]:
            self.acc.x = -ACC
        if pressed_keys[pg.K_RIGHT] or pressed_keys[pg.K_d]:
            self.acc.x = ACC

        self.acc.x += self.vel.x * FRIC
        self.vel += self.acc
        self.pos += self.vel + 0.5 * self.acc

        if self.pos.x > WIDTH:
            self.pos.x = 0
        if self.pos.x < 0:
            self.pos.x = WIDTH
     
        self.rect.midbottom = self.pos

    def jump(self):
        hits = pg.sprite.spritecollide(
            self,
            self.game.platforms,
            False
        )
        if hits and not self.jumping:
            self.jumping = True

            self.vel.y = -15
    
    def cancel_jump(self):
        if self.jumping:
            if self.vel.y < -3:
                self.vel.y = -3

    # Função de limitação por colisão
    def update(self):
        hits = pg.sprite.spritecollide(
            self,
            self.game.platforms,
            False
        )
        
        if self.vel.y > 0 and hits:
            platform = hits[0]

            if self.pos.y < platform.rect.bottom:
                self.vel.y = 0
                self.pos.y = platform.rect.top + 1
                self.jumping = False
                self.pos.x += platform.vel.x*platform.direction
        
    def draw(self, screen):
        screen.blit(self.surf, self.rect)

#### `Platform`

Essa classe define um molde completo de plataforma no jogo, dizendo como ela deve se mover, suas dimensões, etc. Suas funções são:

##### `__init(self, width_max=100, moving=False, speed=0)`

Assim como na classe `Player`, essa função define todas as características principais dessa classe, como dimensões, velocidade, etc.

Aqui, essa função define todas as variáveis que serão usadas e a forma da plataforma. Para tal, a função usa de quatro argumentos, `self`, `width_max`, `moving`, `speed`. 

Args:

> * `self`, como foi mostrado anteriormente, é usada para chamar a classe dentro dela mesma, para poder usar as variáveis de qualquer função;
> * Já `width_max`, que foi inicialmente definida como `100`, é a largura máxima que a plataforma pode assumir, onde, em outras subclasses, é possível alterar esse valor para que a plataforma fique maior ou menor. 
> * O argumento `moving` é uma variável booleana que define se a plataforma irá se mover ou não. Inicialmente ela é definida como `False`.
> * `speed` é um argumento que só será usado se `moving = True`. Ele define a velocidade de movimento da plataforma, que é inicialmente `0`.

Vars:

> * Aqui se define todos os argumentos como acessíveis para toda a classe:
>> ```
>> 8   self.width_max = width_max
>> 9   self.moving = moving
>> 10  self.speed = speed
>> ```
> * `self.direction = random.randint(-1, 1)` define uma direção inicial aleatória para a direção da platafora. Essa variável só será utilizada caso `self.moving = True`.
> * Aqui se define a superfície da plataforma:
>> ```
>> 13  self.surf = pg.Surface(
>> 14      (
>> 15          random.randint(
>> 16              int(width_max/2),
>> 17              int(width_max)
>> 18          ),
>> 19          15
>> 20      )
>> 21  )
>> ```
> Para definir `self.surf`, se define um valor arbitrário para a largura, tal que `width_max/2 <= width <= width_max` e uma altura padrão de `15`.
> * Em `self.surf.fill(green)`, é definida a cor da plataforma como verde.
> * Depois define-se as coordendas da plataforma:
>> ```
>> 25  coordenadas = (
>> 26      random.randint(0, WIDTH-10),
>> 27      random.randint(0, WIDTH-20)
>> 28  )
>> ```
> Aqui, é definida uma variável local de coordenada aleatória, que será usada unicamente para definir as coordenadas da plataforma.
> * Depois é definida a posição e dimensões da caixa limitadora da plataforma - que é idêntica à plataforma:
>> ```
>> 29  self.rect = self.get_rect(
>> 30      center = (coordenadas)
>> 31  )
>> ```
> * Então, em `self.vel = vec(speed, 0)`, é definido o vetor velocidade da plataforma que, nesse caso, é inicialmente `(0, 0)`.
> * `self.pos = vec(coordenadas)`, assim como na classe player, só define o vetor posição da plataforma.

##### `move(self)`

Aqui se define como a plataforma deve se mover, caso ela fizer isso.

Para tal, primeiro se verifica se `self.moving = True`. Então, se isso for verdadeiro, a posição da plataforma é alterada de acordo com usa velocidade e direção atuais com `self.pos.x += self.vel.x*self.direction`.

> ```
> 40  if self.direction == 0:
> 41      self.direction = 1
> 41  if self.pos.x <= 0:
> 42      self.direction = 1
> 43  elif self.pos.x >= WIDTH:
> 44      self.direction = -1
> ```

Esse bloco inverte a direção do movimento, caso a plataforma colida com as paredes e corrige um pequeno erro que pode acontecer, que é a plataforma ficar parada, mesmo que não fosse para ficar.

Já a linha `self.rect.x = self.pos.x` define a posição da caixa delimitadora como o vetor posição da plataforma.

##### `draw(self, screen)`

Essa função só desenha a plataforma na tela.

##### Código

In [188]:
#####################
# Platforms Sprites #
#####################
class Platform(pg.sprite.Sprite):
    def __init__(self, width_max=100, moving=False, speed=0):
        super().__init__()

        self.width_max = width_max
        self.moving = moving
        self.speed = speed
        self.direction = random.randint(-1, 1)
        
        self.surf = pg.Surface(
            (
                random.randint(
                    int(width_max/2),
                    int(width_max)
                ),
                15
            )
        )

        self.surf.fill(green)

        coordenadas = (
            random.randint(0, WIDTH-10),
            random.randint(0, HEIGHT-20)
        )
        self.rect = self.surf.get_rect(
            center = (coordenadas)
        )

        self.vel = vec(speed, 0)
        self.pos = vec(coordenadas)

    def move(self):
        if self.moving:
            self.pos.x += self.vel.x*self.direction

            if self.direction == 0:
                self.direction == 1
            if self.pos.x <= 0:
                self.direction = 1
            elif self.pos.x >= WIDTH:
                self.direction = -1
            
            self.rect.x = self.pos.x
                
    def draw(self, screen):
        screen.blit(self.surf, self.rect)

#### Subclasses de `Platform`

Aqui entra a ideia de subclasse, para reaproveitar a classe `Platform`, alterando somente os argumentos da função `__init()__`.

Em `MinorPlatform(Platform)` é criada uma classe de plataforma que é filha da classe principal `Platform`. Para definir suas características individuais, é usado `super().__init()`, que chama todas as características de `Platform`, mas altera os argumentos de `__init__`.

> ```
> 3   super().__init__(
> 4       width_max=60,
> 5       moving=True,
> 6       speed=3
> 7   )
> ```

Dessa forma foram definedas a largura máxima como `60`, a plataforma como móvel e sua velocidade de movimento como 3.

##### Código

In [189]:
class MinorPlatform(Platform):
    def __init__(self):
        super().__init__(
            width_max=60,
            moving=True,
            speed=3
        )

Em `MajorPlatform(Platform)` foi feita a mesma coisa, mas suas características são:
> * Largura máxima: 80;
> * Móvle;
> * Velocidade: 6.

##### Código

In [190]:
class MajorPlatform(Platform):
    def __init__(self):
        super().__init__(
            width_max=80,
            moving=True,
            speed=6
        )

#### `Enemy`

Classe que define como os inimigos como um todo devem funcionar. Essa classe, assim como `Platform`, define como os inimigos devem funcionar num geral, mas ela pode ser reutilizada facilmente, sendo usada como um molde para qualquer outro inimigo. Suas funções são:

##### `__init__(self, path, speed=0)`

Essa função, assim como nas outras classes, define todas as variáveis e características iniciais da classe.

Para tal, essa função usa dos argumentos `self`, `path` e `speed`, que servem para reutilizar essa classe de maneira útil.

Args:

> * `self`: Como já mencionado, serve para tornar todas as variáveis do tipo `self.var` acessíveis para todas as funções.
> * `path`: Esse argumento é usado para definir o caminho da imagem do inimigo. Esse argumento não é definido inicialmente e deve sempre ser definido nas subclasses.
> * `speed`: É usado assim como nas plataformas. Note que esse argumento é definido inicialmente como `0`.

A partir disso, são definidas todas as características e variáveis principais da classe.

Vars:

> * `self.image = pg.image.load(path)`: aqui é definida a superfície do inimigo como uma imagem, que deve ser chamada a partir da variável `path`.
> * `self.rect = self.image.get_rect()`: Aqui é definida a caixa delimitadora a partir das dimensões da imagem do inimigo.
> * Em `self.direction = random.randint(-1, 1)` é definida uma direção inicial aleatória para o movimento do inimigo.
> * `self.pos = vec((random.randint(0, WIDTH), 0))` define o vetor posição do inimigo, definindo a posição em `x` de maneira aleatória e a posição em `y` como `0`, ou seja, o inimigo "nasce" - ou "spawna", para os mais íntimos - no topo da tela.
> * `self.vel = vec(speed, 0)` define o vetor velocidade que fará o inimigo se mover em `x`.<br>
_Aqui gostaria de pontuar que queria alterar isso para poder deixar essa classe mais versátil, permitindo inimigos com diferentes movimentações, mas, por enquanto, vou deixar assim como uma Demo._

##### `move(self)`

Essa função é idêntica a função `move()` da classe `Platform`.

##### `draw(self)`

Essa função só desenha o inimigo na tela.

##### Código

In [191]:
###########
# Enemies #
###########
class Enemy(pg.sprite.Sprite):
    def __init__(self, path, speed=0):
        super().__init__()
        self.image = pg.image.load(path)
        self.rect = self.image.get_rect()
        self.direction = random.randint(-1, 1)

        self.pos = vec((random.randint(0, WIDTH), 0))
        self.vel = vec(speed, 0)

    def move(self):
        self.pos.x += self.vel.x*self.direction

        if self.direction == 0:
            self.direction == 1
        if self.pos.x <= 0:
            self.direction = 1
        elif self.pos.x >= WIDTH:
            self.direction = -1

        self.rect.x = self.pos.x
        self.rect.y = self.pos.y

    def draw(self, screen):
        screen.blit(self.image, self.rect)

#### Subclasses de `Enemy`

Essa subclasse reutiliza a classe `Enemy`.

##### Código

In [192]:
class BasicEnemy(Enemy):
    def __init__(self):
        super().__init__(
            path="pixel enemy.png",
            speed=4
        )

### `Game`

Essa classe contém todo o funcionamento, desde a inicialização e rederização, até a criação e manipulação de cada sprite.
Suas funções são:

#### `__init__(self)`

Essa função define todas as variáveis que serão usadas durante o jogo

O único argumento que ela recebe é `self`, já explicado anteriormente.

Vars:

> * `self.running = True` define o estado do jogo - ligado ou desligado. Como ele está como True, o jogo estará rodadndo.
> * `self.screen = pg.display.set_mode((WIDTH, HEIGHT))` define a tela do jogo, com as dimensões especificadas anteriormente.
> * `self.clock = pg.time.Clock()` define o tempo do jogo, permitindo controlar a taxa de FPS.
> * `self.camera_offset = 0` cria o offset da câmera do jogo, que será posteriormente usado para dar o efeito de que a câmera sobe.
> * `world_height = 0` define a altura inicial do mundo como 0. Ela aumenta conforme a camera sobe.
> * `camera_run = False` define que a camera inicialmente fica parada. Essa variável será definida como `True` quando a camera for subir.
> * `self.all_sprites = pg.sprite.Group()` cria um grupo para todas as sprites que serão criadas no jogo, incluindo o jogador.
> * `self.platforms = pg.sprite.Group()` cria um grupo para colocar todas as plataformas que serão criadas - plataformas também são consideradas sprites, então também ficarão em `self.all_sprites`.
> * `self.enemies = pg.sprite.Group()` cria um grupo para os inimigos.
> * `self.setup()` chama essa função, que será definida depois.

Os grupos são muito importantes pois permitem manipular as sprites de maneira controlada e organisar cada grupo.

#### `setup(self)`

Essa função cria literalmente o setup do jogo, criando o player, a plataforma base - onde o player "nascerá" - e mais 5 plataformas iniciais.

> ```
> 21  self.player = Player(self)
> 22  base_platform = Platform()
> ```
Esse trecho cria o player e a plataforma base a partir das classes, definidas anteriormente. `base_platform` não recebe o argumento `self` porque não será usada posteriormente, a não ser no grupo `self.platform`, que permite manipular todas as plataformas de uma vez.

> ```
> 24  base_platform.surf = pg.Surface((WIDTH, 20))
> 25  base_platform.surf.fill(red)
> ```
Esse bloco cria a superfície da plataforma base - que será diferente das outras - e a colore de vermelho.

```
27  base_platform.rect = base_platform.surf.get_rect(
28      center=(WIDTH/2, WIDTH-5)
29  )
```
Aqui cria-se o `rect` da plataforma.

> ```
> 31  self.platforms.add(base_platform)
> 32
> 33  self.all_sprites.add(base_platform)
> 34  self.all_sprites.add(self.player)
> ```
Aqui todas as novas sprites são adicionadas aos seus respectivos grupos.

> ```
> 36  for _ in range(5):
> 37      while True:
> 38          pl = Platform()
> 39          # Se a plataforma for correta...
> 40          if not self.check_collision(pl):
> 41              break
> 42
> 43      self.platforms.add(pl)
> 44      self.all_sprites.add(pl)
> ```

Aqui são criadas as 5 plataformas iniciais. Primeiro o código entra no `for`, que vai criar as 5 plataformas. Então ele entra no `while`, que irá criar uma plataforma com `pl = Platform()` e checar se ela colide com alguma outra plataforma, para que as plataformas não fiquem sobrepostas. Caso não haja colisão, ele sai do `while` e adiciona a nova plataforma aos grupos.

#### `check_collision(self, sprite)`

Essa função tem os argumentos `self` e `sprite`. `sprite` é a sprite específica que a função irá usar. Ela checa se há colisão da sprite com alguma plataforma qualquer ou se ela está muito próxima de outra plataforma.

> ```
> 47  if pg.sprite.spritecollide(sprite, self.platforms, False):
> 48      return True
> ```

Este trecho verifica se há colisão.

> ```
> 49  else:
> 50      for entity in self.platforms:
> 51          if entity == sprite:
> 52              continue
> 53          if (
> 54              abs(sprite.rect.top - entity.rect.bottom) < 50
> 55              and
> 56              abs(sprite.rect.bottom - entity.rect.top) < 50
> 57          ):
> 58              return True
> 59  return False
> ```

Aqui, caso não haja colisão, ele vai verificar cada entidade em `self.platform`, e analisar, primeiro, se a entidade é a própria sprite - essa parte é importante porque, como as posições, nesse caso, são iguais, ele sempre irá acusar colisão - então ele verifica se a sprite está distanciada em, no mínimo 50 pixels das plataformas. Caso nenhuma condição seja satisfeita, a função retorna `False`, pois não há colisões.

#### `plat_gen(self)`

Essa função cria as novas plataformas.

O `while` cria plataformas até que hajam, no máximo, 6 delas. Para tal, o código usa a variável `PlatformClass`, que será usada para definir qual tipo de plataforma será usada

O `while True:` é o mesmo do setup, que irá criar as plataformas e verificar se elas são válidas. Se não forem, a função a destrói e reinicia a criação dessa plataforma, caso dê certo, ele adiciona a plataforma aos grupos:

> ```
> 66  if self.world_height <= 1000:
> 67      PlatformClass = Platform
> ```

Esse `if` define a classe que será usada como a de uma plataforma comum, caso a altura do mundo ainda seja baixa (<= 1000).

> ```
> 68  elif self.world_height > 1000 and self.world_height <= 3000:
> 69      PlatformClass = random.choice(
> 70          [Platform, MinorPlatform]
> 71      )
> ```

Aqui é definido o tipo de classe que será usada, caso a altura do mundo esteja entre 1000 e 3000. Nesse caso, é escolhida de forma aleatória entre a classe padrão e a classe `MinorPlatform`.

> ```
> 72  elif self.world_height > 3000 and self.world_height <= 9000:
> 73      PlatformClass = random.choice(
> 74          [Platform, MinorPlatform, MajorPlatform]
> 75      )
> ```

Caso a altura esteja entre 3000 e 9000, as possíveis escolhas para `PlatformClass` são `Platform`, `MinorPlatform` e `MajorPlatform`.

> ```
> 76  else:
> 77      chances = [1, 3, 6]
> 78      PlatformClass = random.choices(
> 79          [Platform, MinorPlatform, MajorPlatform],
> 80          weights=chances
> 81      )[0]
> ```

Caso a altura seja maior que 9000, as possíveis escolhas de classe são as mesmas, mas com pesos diferentes, onde `Platform` tem uma chance de 10%, `MinorPlatform` tem 30% e `MajorPlatform` tem 60% de chance. Isso adiciona mais dificuldade ao jogo. Para isso, o código seleciona o primeiro item da lista que o método `choices` escolhe.

> ```
> 83  p = PlatformClass()
> 84  p.rect.center = (
> 85      random.randrange(0, WIDTH),
> 86      -(random.randrange(0, 50))
> 87  )
> 88
> 89  p.pos = vec(p.rect.center)
> ```

Aqui, finalmente, a plataforma é criada a partir da classe escolhida e seu centro é definido de maneira aleatória, com limitação de altura, podendo ir de 0 até -50. Ou seja, a plataforma é criada fora da tela, antes de ser mostrada na tela. Então um vetor posição é criado, baseado em seu `rect.center`.

Agora só falta verificar se essa é uma plataforma válida!

> ```
> 91  if (
> 92      not self.check_collision(p)
> 93      and
> 94      p.rect.top - self.player.rect.bottom <= 80
> 95  ):
> 96      break
> ```

Aqui ele verifica se há sobreposição de plataformas e se a plataforma está a até 80 pixels de distância do player, pois, se não, não seria possível alcançar a plataforma com um pulo. Caso esteja tudo certo, ele sai do `while` de criação.

Ao sair do primeiro `while`, a função adiciona a nova sprite aos grupo com `self.platforms.add(p)` e `self.all_sprites.add(p)`.

#### `enemy_gen(self)`

Essa função, assim como a `plat_gen()`, cria sprites. A diferença é que, nesse caso, ela cria inimigos.

Essa função, por mais que crie sprites, assim como `plat_gen()`, é muito mais simples, pois não é tão relevante o inimigo estar sobreposto a uma plataforma. Algo que gostaria de comentar é sobre os `print()` que usei nessa função, pois os utilizei para depurar o código e queria deixar um pouco dessa marca, para mostrar parte do processo de refinamento.

#### `input(self)`

Essa função lê todos os inputs extras do jogo.

Para isso, foi usado `pg.event.get()`, que reconhece todo evento que acontece - eventos, nesse caso, são descritos como interações entre o usuário e a máquina, ou seja, basicamente um input, mas ele possibilita ler teclas, por exemplo.

> ```
> 114 if event.type == pg.QUIT:
> 115     self.running = False
> 116 if event.type == pg.KEYDOWN:
> 117     if event.key == pg.K_SPACE:
> 118         self.player.jump()
> 119 if event.type == pg.KEYUP:
> 120     if event.key == pg.K_SPACE:
> 121         self.cancel_jump()
> ```

`event.type` é o tipo de evento que está acontecendo.

inputs:

> * `pg.QUIT` significa clicar no botão de "X" da janela, o que significa que o usuário quer sair do jogo, então o código define `self.running = False`, ou seja, o jogo não é mais para rodar.
> * `pg.KEYDOWN` significa que o usuário está apertando uma tecla - vale ressaltar que clicar e apertar são coisas diferentes: clicar significa que o código lerá somente quando a tecla desce ou, em outros casos, quando ela sobe, e apertar significa que o código lerá o pressionamento da tecla enquanto isso estiver ocorrendo.
>> * Se o usuário está apertando alguma tecla, ele reconhece qual é a partir de `event.key`, caso a tecla seja espaço, ele chama `self.player.jump()`, ou seja, irá fazer o player pular.
>> * Porém, se o usuário soltar a tecla, o programa irá reconhecer através de `pg.KEYUP` e irá chamar `self.cancel_jump()`, cancelando o pulo.

#### `update(self)`

Essa função atualiza as informações do jogo.

> * `self.player.move()` faz com que o player sempre execute essa função, ou seja, ele sempre possa se mover. O mesmo serve para `self.player.update()`.
> * `hits = pg.sprite.spritecollide(self.player, self.enemies, False)` lê qualquer hit que um inimigo dê no player.
> ```
> 128 if hits:
> 129     self.player.life -= 2
> ```
> * Isso faz com que, caso o player sofra um hit, ele perca vida.
> ```
> 131 for plat in self.platforms:
> 132     plat.move()
> 133
> 134     if plat.rect.top >= HEIGHT:
> 135         plat.kill()
> 136
> 137 for enemy in self.enemies:
> 138     enemy.move()
> 139         
> 140     if enemy.rect.top >= HEIGHT:
> 141         enemy.kill()
> ```
> * Aqui o programa executa a função `move()` de cada sprite e, caso qualquer uma delas ultrapasse a parte inferior da tela, o código deleta a que ultrapassou o limite da tela - safadinha ela, hein - através de `plat.kill()`, ou `enemy.kill()`.
> ```
> 143 # Camera
> 144     # Se o player ultrapassar 2/3 da tela
> 145 if self.player.rect.top < HEIGHT/3:
> 146     self.camera_run = True
> 147     self.camera_offset = abs(self.player.vel.y)
> 148     self.world_height += self.camera_offset
> 149     self.player.pos.y += self.camera_offset
> 150
> 151     for enemy in self.enemies:
> 152         enemy.pos.y += self.camera_offset
> 153     for plat in self.platforms:
> 154         plat.rect.y += self.camera_offset
> 155
> 156 elif self.camera_run:
> 157     self.camera_offset = 1
> 158     self.world_height += self.camera_offset
> 159     self.player.pos.y += self.camera_offset
> 160     
> 161     for enemy in self.enemies:
> 162         enemy.pos.y += self.camera_offset
> 163     for plat in self.platforms:
> 164         plat.rect.y += self.camera_offset
> ```
> * Aqui são definidos os efeitos da câmera, onde `self.camera_offset` é a velocidade de avanço da câmera. A partir disso, se o primeiro `if` for verdadeiro, ela inicia o scroll da câmera com `self.camera_run = True` e define `self.camera_offset = abs(self.player.vel.y)`, ou seja, ela avança junto com o jogador. Agora, caso a primeira condição não seja satisfeita, mas a segunda sim, ele define `self.camera_offset = 1`. Para dar esse efeito de sroll, o código soma velocidade da câmera a posição em `y` de cada sprite, movendo, na realidade, as sprites. Ademais, isso atualiza a altura do mundo.
> * Depois ele cria as plataformas e inimigos com `self.plat_gen()` e `self.enemy_gen`.
> * Por fim ele define a condição de Game Over, que é se o player cair ou zerar a vida.

#### `draw(self)`

Essa função "desenha" tudo na tela, de fonte até sprites.

> * `self.screen.fill(black)` colore a tela de preto.
> ```
> 181 text_height = small_font.render(
> 182     f"Height: {int(self.world_height/10)}",
> 183     True,
> 184     white
> 185 )
> ```
> * Esse trecho define o texto da altura do jogo, colorindo-o de branco.
> ```
> 186 text_lifes = small_font.render(
> 187     f"Hp: {self.player.life}",
> 188     True,
> 189     green
> 190 )
> ```
> * Assim como o anterior, esse trecho cria um texto, mas, nesse caso, são as vidas, em verde.
> * Então os textos são desenhados na tela a partir de `self.screen.blit(text_height, (10, 10))`, que desenha `text_height` na posição `(10, 10)`, e `self.screen.blit(text_lifes, (10, 30))`, que desenha `text_lifes` na posição `(10, 30)`.
> ```
> 195 for entity in self.all_sprites:
> 196     entity.draw(self.screen)
> ```
> * Aqui todas as sprites são desenhadas na tela.
> * Por fim, `pg.display.update()` atualiza toda a tela, sempre.

#### `run(self)`

Contém o loop do jogo inteiro!! No loop, é definida uma taxa de renderização de 60 FPS com `self.clock.tick(60)` e o loop chama todas as funções relevantes para o jogo, `self.input()`, `self.update()` e `self.draw()`. Ao sair do `while`, o jogo para com `pg.quit()` e `sys.exit()`.

##### Código

In [193]:
################################################################
#----------------------------Game------------------------------#
################################################################
class Game:
    def __init__(self):
        self.running = True

        self.screen = pg.display.set_mode((WIDTH, HEIGHT))
        self.clock = pg.time.Clock()
        self.camera_offset = 0
        self.world_height = 0
        self.camera_run = False

        self.all_sprites = pg.sprite.Group()
        self.platforms = pg.sprite.Group()
        self.enemies = pg.sprite.Group()

        self.setup()

    def setup(self):
        self.player = Player(self)
        base_platform = Platform()

        base_platform.surf = pg.Surface((WIDTH, 20))
        base_platform.surf.fill(red)

        base_platform.rect = base_platform.surf.get_rect(
            center=(WIDTH/2, WIDTH-5)
        )
        
        self.platforms.add(base_platform)
        
        self.all_sprites.add(base_platform)
        self.all_sprites.add(self.player)

        for _ in range(5):
            while True:
                pl = Platform()
                # Se a plataforma for correta...
                if not self.check_collision(pl):
                    break

            self.platforms.add(pl)
            self.all_sprites.add(pl)

    def check_collision(self, sprite):
        if pg.sprite.spritecollide(sprite, self.platforms, False):
            return True
        else:
            for entity in self.platforms:
                if entity == sprite:
                    continue
                if (
                    abs(sprite.rect.top - entity.rect.bottom) < 50
                    and
                    abs(sprite.rect.bottom - entity.rect.top) < 50
                ):
                    return True
    
        return False

    def plat_gen(self):
        while len(self.platforms) < 6:
            while True:

                if self.world_height <= 1000:
                    PlatformClass = Platform
                elif self.world_height > 1000 and self.world_height <= 3000:
                    PlatformClass = random.choice(
                        [Platform, MinorPlatform]
                    )
                elif self.world_height > 3000 and self.world_height <= 9000:
                    PlatformClass = random.choice(
                        [Platform, MinorPlatform, MajorPlatform]
                    )
                else:
                    chances = [1, 3, 6]
                    PlatformClass = random.choices(
                        [Platform, MinorPlatform, MajorPlatform],
                        weights=chances
                    )[0]

                p = PlatformClass()
                p.rect.center = (
                    random.randrange(0, WIDTH),
                    -(random.randrange(0, 50))
                )

                p.pos = vec(p.rect.center)

                if (
                    not self.check_collision(p)
                    and
                    p.rect.top - self.player.rect.bottom <= 80
                ):
                    break

            self.platforms.add(p)
            self.all_sprites.add(p)

    def enemy_gen(self):
        if self.world_height >= 500 and len(self.enemies) == 0:
            print("entrou na função")
            
            enemy = BasicEnemy()
            print("criando...")
             
            self.enemies.add(enemy)
            self.all_sprites.add(enemy)
            print("adicionado")

    def input(self):
        for event in pg.event.get():
            if event.type == pg.QUIT:
                self.running = False
            if event.type == pg.KEYDOWN:
                if event.key == pg.K_SPACE:
                    self.player.jump()
            if event.type == pg.KEYUP:
                if event.key == pg.K_SPACE:
                    self.player.cancel_jump()

    def update(self):
        self.player.move()
        self.player.update()

        hits = pg.sprite.spritecollide(self.player, self.enemies, False)
        if hits:
            self.player.life -= 5

        for plat in self.platforms:
            plat.move()

            if plat.rect.top >= HEIGHT:
                plat.kill()

        for enemy in self.enemies:
            enemy.move()
            
            if enemy.rect.top >= HEIGHT:
                enemy.kill()

        # Camera
            # Se o player ultrapassar 2/3 da tela
        if self.player.rect.top <= HEIGHT/3:
            self.camera_run = True
            self.camera_offset = abs(self.player.vel.y)
            self.world_height += self.camera_offset
            self.player.pos.y += self.camera_offset

            for enemy in self.enemies:
                enemy.pos.y += self.camera_offset
            for plat in self.platforms:
                plat.rect.y += self.camera_offset
        
        elif self.camera_run:
            self.camera_offset = 1
            self.world_height += self.camera_offset
            self.player.pos.y += self.camera_offset

            for enemy in self.enemies:
                enemy.pos.y += self.camera_offset
            for plat in self.platforms:
                plat.rect.y += self.camera_offset
        
        # Criando plataformas e inimigo
        self.plat_gen()
        
        self.enemy_gen()

        # Game Over
        if (
            self.player.rect.top >= HEIGHT
            or
            self.player.life <= 0
        ):
            self.running = False

    def draw(self):
        self.screen.fill(black)
        text_height = small_font.render(
            f"Height: {int(self.world_height/10)}",
            True,
            white
        )
        text_lifes = small_font.render(
            f"Hp: {self.player.life}",
            True,
            green
        )

        self.screen.blit(text_height, (10, 10))
        self.screen.blit(text_lifes, (10, 30))

        for entity in self.all_sprites:
            entity.draw(self.screen)
        
        pg.display.update()

    def run(self):
        while self.running:
            self.clock.tick(FPS)
            self.input()
            self.update()
            self.draw()
        
        pg.quit()
        sys.exit()

game = Game()
game.run()

entrou na função
criando...
adicionado
entrou na função
criando...
adicionado


SystemExit: 